# Subbasin attribute exploration
1426 basins; 32802 subbasin records; 1101 subbasin attributes. Training-only screening; no PCA fit or LSTM training yet.

In [1]:
from pathlib import Path
import csv,json,html
from IPython.display import display,HTML
B=Path.cwd()
def read(name):
 with open(B/name,newline='') as f:return list(csv.DictReader(f))
def table(rows,cols):
 text='<table><tr>'+''.join('<th>'+html.escape(c)+'</th>' for c in cols)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(c,'')))+'</td>' for c in cols)+'</tr>' for r in rows)+'</table>'
 display(HTML(text))
summary=json.loads((B/'exploration_summary.json').read_text())
print(json.dumps(summary,indent=2))

{
  "phase": "exploration_complete",
  "basins": 1426,
  "training_basins": 1326,
  "unseen_basins": 100,
  "subbasin_records": 32802,
  "training_subbasin_records": 30604,
  "unseen_subbasin_records": 2198,
  "subbasin_attributes": 1101,
  "whole_basin_attributes": 1178,
  "whole_basin_only_attributes": 80,
  "subbasin_count_min": 1,
  "subbasin_count_max": 227,
  "single_subbasin_basins": 304,
  "area_sum_relative_error_max": 2.6771150564519666e-07,
  "categories": {
    "Climate": 625,
    "Land cover": 74,
    "Topography": 20,
    "Soil": 352,
    "Open water": 22,
    "Geology": 8
  },
  "sources": {
    "RDRS": 373,
    "WorldClim": 228,
    "WorldClim (derived, Oudin et al., 2005)": 24,
    "MCD15A2H.061": 24,
    "GLCLUC 2000-2020": 8,
    "GLCLU 2019": 20,
    "MCD12Q1.061": 18,
    "LGRIP30": 4,
    "MERIT Hydro": 15,
    "MERIT Hydro Basins": 3,
    "MERIT Hydro, MERIT Hydro Basins": 2,
    "Pelletier": 16,
    "SOILGRIDS": 336,
    "HydroLAKES": 22,
    "GLHYMPS": 8
  },
 

## Provisional physical-property shortlist
317 candidate attributes: 208 soil, 64 land-cover, 11 topographic, 20 open-water and 8 geology. See feature_reduction_plan.csv for each proposed decision. Domain balance, transformations and correlation review are still required before PCA.

## Attribute inventory: training-only statistics

In [2]:
inventory=read('attribute_inventory_and_screening.csv')
table(inventory[:25],['attribute','category','unit','source','missing_fraction_basin_area_weighted','domain_role','quality_action'])

attribute,category,unit,source,missing_fraction_basin_area_weighted,domain_role,quality_action
num_years_rdrs,Climate,years,RDRS,0.0,exclude_record_length,review
PR0_mean,Climate,mm year-1,RDRS,0.0,climate_variant_review,review
PR0_std,Climate,mm year-1,RDRS,0.0,climate_variant_review,review
pet1_mean,Climate,mm year-1,RDRS,0.0,climate_variant_review,review
pet1_std,Climate,mm year-1,RDRS,0.0,climate_variant_review,review
TT_mean,Climate,C,RDRS,0.0,climate_variant_review,review
TT_std,Climate,C,RDRS,0.0,climate_variant_review,review
aridity1_mean,Climate,-,RDRS,0.0,climate_variant_review,review
aridity1_std,Climate,-,RDRS,0.0,climate_variant_review,review
fracsnow1_mean,Climate,-,RDRS,0.0,climate_variant_review,review


## Basic screening candidates

In [3]:
flagged=[r for r in inventory if r['quality_action']!='review']
table(flagged[:80],['attribute','quality_action','duplicate_of','training_min','training_max'])
print('Flagged attributes:',len(flagged))

attribute,quality_action,duplicate_of,training_min,training_max
tdmin_std_month_01,review_exact_duplicate,tdmin_mean_month_01,-35.365516662597656,9.949698448181152
tdmin_std_month_02,review_exact_duplicate,tdmin_mean_month_02,-34.606483459472656,11.444262504577637
tdmin_std_month_03,review_exact_duplicate,tdmin_mean_month_03,-33.663291931152344,13.404473304748535
tdmin_std_month_04,review_exact_duplicate,tdmin_mean_month_04,-26.51444435119629,17.07618522644043
tdmin_std_month_05,review_exact_duplicate,tdmin_mean_month_05,-18.696422576904297,21.03822135925293
tdmin_std_month_06,review_exact_duplicate,tdmin_mean_month_06,-12.711496353149414,23.786758422851562
tdmin_std_month_07,review_exact_duplicate,tdmin_mean_month_07,-9.395474433898926,24.036579132080078
tdmin_std_month_08,review_exact_duplicate,tdmin_mean_month_08,-11.818901062011719,24.030147552490234
tdmin_std_month_09,review_exact_duplicate,tdmin_mean_month_09,-18.560361862182617,21.91009521484375
tdmin_std_month_10,review_exact_duplicate,tdmin_mean_month_10,-27.09076499938965,19.045455932617188


Flagged attributes: 25


## Subbasin granularity and area checks

In [4]:
basins=read('basin_subbasin_audit.csv')
table(sorted(basins,key=lambda r:abs(float(r['area_relative_difference'])),reverse=True)[:20],['basin_id','catchment_split','subbasins','area_sum_km2','lumped_area_km2','area_relative_difference','effective_subbasins','single_subbasin'])

basin_id,catchment_split,subbasins,area_sum_km2,lumped_area_km2,area_relative_difference,effective_subbasins,single_subbasin
CAN_05EE006,training_donor,1,0.4637890991616448,0.463788975,2.6771150564519666e-07,1.0,1
CAN_01AL004,training_donor,1,1.737233343919142,1.737232963,2.1926773790256175e-07,1.0,1
CAN_08HA068,unseen_test,1,2.017671306853968,2.017671102,1.0152991129624904e-07,1.0,1
CAN_08NM242,training_donor,1,3.683181370970033,3.683181572,-5.4580520391272547e-08,1.0,1
USA_01542810,training_donor,1,13.397850916469094,13.39785025,4.974448009098351e-08,1.0,1
CAN_01BP002,training_donor,1,25.10301599117919,25.10301507,3.669595812487046e-08,1.0,1
USA_09035800,training_donor,1,23.007039026219214,23.00703834,2.982649060985564e-08,1.0,1
CAN_01BU009,training_donor,1,5.7947295407713915,5.794729373,2.8952411863752622e-08,1.0,1
CAN_02GA043,training_donor,1,13.941835560527956,13.94183516,2.872849600458747e-08,1.0,1
USA_09066300,training_donor,1,15.260502221780337,15.26050262,-2.6094793427364055e-08,1.0,1


## Highly correlated attribute pairs: review, not automatic removal

In [5]:
pairs=read('high_correlation_review_pairs.csv')
table(sorted(pairs,key=lambda r:abs(float(r['spearman_r'])),reverse=True)[:40],['attribute_1','attribute_2','spearman_r','category_1','category_2'])

attribute_1,attribute_2,spearman_r,category_1,category_2
forest_height_2000_min,forest_height_2020_min,1.0000000000000084,Land cover,Land cover
fracsnow2_mean_month_06,fracsnow2_std_month_06,1.0000000000000056,Climate,Climate
reservoir_area_min,reservoir_area_mean,1.0000000000000049,Open water,Open water
reservoir_area_min,reservoir_area_max,1.0000000000000049,Open water,Open water
reservoir_area_min,reservoir_area_total,1.0000000000000049,Open water,Open water
reservoir_area_mean,reservoir_area_max,1.0000000000000049,Open water,Open water
reservoir_area_mean,reservoir_area_total,1.0000000000000049,Open water,Open water
reservoir_area_max,reservoir_area_total,1.0000000000000049,Open water,Open water
reservoir_volume_min,reservoir_volume_mean,1.0000000000000049,Open water,Open water
reservoir_volume_min,reservoir_volume_max,1.0000000000000049,Open water,Open water


## Reproduce the full audit
Run on a CPU compute session with NumPy and access to the original CAMELS-SPAT archive. The source is embedded below. RUN_FULL_AUDIT=False only reads existing reports. Setting True recomputes outputs in this experiment folder and may take several minutes.

In [6]:
AUDIT_SOURCE='from pathlib import Path\nimport csv,json,time,hashlib,collections,os\nimport numpy as np\nR=Path(\'/project/6102174/data/misc-data/CAMELS-SPAT\');B=Path.home()/\'Desktop/LSTM_RR_Modelling/Subbasin_Heterogeneity_PCA\';B.mkdir(exist_ok=True)\ndef savecsv(name,rows,fields=None):\n with open(B/name,\'w\',newline=\'\') as f:w=csv.DictWriter(f,fieldnames=fields or list(rows[0]));w.writeheader();w.writerows(rows)\ndef savejson(name,obj):(B/name).write_text(json.dumps(obj,indent=2))\nwith open(B.parent/\'Model A Lumped (Daymet)/split_manifest.csv\') as f:manifest=list(csv.DictReader(f))\nassert len(manifest)==1426 and collections.Counter(r[\'split\'] for r in manifest)=={\'training_donor\':1326,\'unseen_test\':100}\nwith open(R/\'attributes/attributes-lumped.csv\') as f:\n reader=csv.reader(f);lhead=next(reader);lmeta=[];larea={}\n for row in reader:\n  lmeta.append(row[:4])\n  if row[1]==\'basin_area\':larea={b:float(v) for b,v in zip(lhead[4:],row[4:])}\npaths={p.name.removesuffix(\'_attributes.csv\'):p for p in (R/\'attributes\').glob(\'*/*_attributes.csv\')}\nassert set(paths)=={r[\'basin_id\'] for r in manifest}\nnonnumeric=collections.defaultdict(collections.Counter)\nmatrices=[];indices=[];basins=[];schema=None;bad=[];masked=[];start=time.time();cursor=0\nfor k,m in enumerate(manifest):\n bid=m[\'basin_id\'];p=paths[bid]\n with open(p) as f:reader=csv.reader(f);h=next(reader);rows=list(reader)\n meta=[r[:4] for r in rows];ids=h[4:]\n if schema is None:schema=meta\n assert meta==schema,(bid,\'attribute schema/order changed\')\n assert len(ids)==len(set(ids)) and len(ids)>0,(bid,\'duplicate/empty subbasin IDs\')\n values=np.full((len(ids),len(meta)),np.nan,dtype=\'float64\')\n for j,row in enumerate(rows):\n  assert len(row)==len(h),(bid,j,\'ragged row\')\n  for i,v in enumerate(row[4:]):\n   try:values[i,j]=float(v)\n   except ValueError:\n    if v.strip()==\'--\':masked.append(dict(basin_id=bid,subbasin_id=ids[i],catchment_split=m[\'split\'],attribute=row[1],value=v))\n    if v.strip().lower() not in (\'\',\'nan\',\'na\',\'n/a\',\'none\',\'null\',\'--\'):\n     bad.append(dict(basin_id=bid,subbasin_id=ids[i],catchment_split=m[\'split\'],attribute=row[1],value=v))\n     if m[\'split\']==\'training_donor\':nonnumeric[row[1]][v]+=1\n values[~np.isfinite(values)]=np.nan\n jarea=next(j for j,s in enumerate(meta) if s[1]==\'basin_area\');area=values[:,jarea].astype(\'float64\')\n assert np.isfinite(area).all() and (area>0).all(),(bid,\'invalid subbasin areas\')\n weights=area/area.sum();train=m[\'split\']==\'training_donor\';n=len(ids)\n for i,sub in enumerate(ids):indices.append(dict(basin_id=bid,subbasin_id=sub,catchment_split=m[\'split\'],seen_unseen=\'Seen\' if train else \'Unseen\',area_km2=float(area[i]),within_basin_area_weight=float(weights[i]),pca_fit_weight=float(weights[i]/1326) if train else 0.))\n basins.append(dict(basin_id=bid,catchment_split=m[\'split\'],basin_group=m[\'basin_group\'],subbasins=n,start_row=cursor,end_row_exclusive=cursor+n,area_sum_km2=float(area.sum()),lumped_area_km2=larea[bid],area_relative_difference=float((area.sum()-larea[bid])/larea[bid]),effective_subbasins=float(1/np.sum(weights**2)),single_subbasin=int(n==1)))\n cursor+=n;matrices.append(values)\n if (k+1)%100==0:print(\'SCANNED\',k+1,\'BASINS\',cursor,\'SUBBASINS\',flush=True)\nX=np.concatenate(matrices);del matrices\nsavecsv(\'masked_numeric_values.csv\',masked,[\'basin_id\',\'subbasin_id\',\'catchment_split\',\'attribute\',\'value\'])\nsavecsv(\'categorical_attribute_values.csv\',bad,[\'basin_id\',\'subbasin_id\',\'catchment_split\',\'attribute\',\'value\'])\nsavecsv(\'subbasin_index.csv\',indices);savecsv(\'basin_subbasin_audit.csv\',basins)\nnp.savez_compressed(B/\'subbasin_attribute_matrix.npz\',values=X,attributes=np.array([s[1] for s in schema]))\ntrainmask=np.array([r[\'seen_unseen\']==\'Seen\' for r in indices]);T=X[trainmask];W=np.array([r[\'pca_fit_weight\'] for r in indices])[trainmask];assert np.isclose(W.sum(),1.)\nfitids=[r[\'basin_id\'] for r in manifest if r[\'split\']==\'training_donor\'];holdids=[r[\'basin_id\'] for r in manifest if r[\'split\']==\'unseen_test\']\nassert len(fitids)==1326 and len(holdids)==100\nlookup={(r[\'basin_id\'],r[\'subbasin_id\']):float(r[\'pca_fit_weight\']) for r in indices}\ncat_mass=collections.Counter()\nfor r in bad:\n if r[\'catchment_split\']==\'training_donor\':cat_mass[r[\'attribute\']]+=lookup[(r[\'basin_id\'],r[\'subbasin_id\'])]\n\ninventory=[];duplicate_hashes={};duplicate_groups=[]\nfor j,(cat,name,unit,source) in enumerate(schema):\n v=T[:,j];good=np.isfinite(v);n=int(good.sum());a=v[good];p=W[good];numeric_missing=float(1-p.sum());missing=float(np.clip(numeric_missing-cat_mass[name],0.,1.));mean=float(np.sum(a*p)/p.sum()) if n else np.nan;sd=float(np.sqrt(np.sum(p*(a-mean)**2)/p.sum())) if n else np.nan\n mn=float(a.min()) if n else np.nan;mx=float(a.max()) if n else np.nan;span=mx-mn;exact=bool(n and mn==mx);near=bool(n and span<=1e-12*max(1.,abs(mn),abs(mx)))\n flags=[];role=\'physical_candidate\';reason=\'Review physical meaning and redundancy; retain variability measures.\'\n if cat==\'Climate\':role=\'climate_variant_review\';reason=\'Separate climate from primary physical-property PCA; RDRS statistics may span beyond training years.\'\n if name.startswith(\'num_years\'):role=\'exclude_record_length\';reason=\'Record availability, not a physical catchment property.\'\n if name==\'basin_area\':role=\'area_weight_only\';reason=\'Use for area weights; avoid encoding subbasin discretization in primary PCA.\'\n if name in (\'centroid_lat\',\'centroid_lon\'):role=\'location_context_only\';reason=\'Geographic position is separate from physical-property heterogeneity.\'\n if \'aspect\' in name:role=\'circular_variable_review\';reason=\'Angles need circular treatment; do not enter raw degree values into linear PCA.\'\n if \'uncertainty\' in name.lower() or \'no_data\' in name or \'unclassified\' in name:role=\'data_quality_review\';reason=\'Data coverage or estimation uncertainty; retain as diagnostics before deciding on predictors.\'\n if \'fraction\' in name:flags.append(\'compositional_fraction_review\')\n if name in (\'open_water_number\',\'known_reservoirs\') or name.endswith(\'_total\'):flags.append(\'extensive_variable_area_normalization_review\')\n if cat==\'Soil\':flags.append(\'preserve_depth_and_spatial_statistic_meanings\')\n if name.endswith(\'_std\'):flags.append(\'potential_within_subbasin_heterogeneity_descriptor\')\n if \'log_permeability\' in name:flags.append(\'already_log_transformed_do_not_log_again\')\n action=\'review\'\n if nonnumeric[name]:action=\'review_categorical_encoding\';role=\'categorical_season_review\';reason=\'Non-numeric categories must be encoded appropriately before PCA; not numeric missing values.\'\n elif not n:action=\'exclude_all_missing\'\n elif exact:action=\'exclude_constant_training\'\n elif near:action=\'review_numerically_near_constant\'\n elif missing>.4:action=\'review_high_missingness\'\n duplicate_of=\'\'\n if n and not exact:\n  vv=v.copy();vv[np.isnan(vv)]=np.nan;vv[vv==0]=0;digest=(unit,hashlib.sha256(vv.tobytes()).hexdigest())\n  if digest in duplicate_hashes:duplicate_of=duplicate_hashes[digest];duplicate_groups.append(dict(attribute=name,duplicate_of=duplicate_of,unit=unit));action=\'review_exact_duplicate\'\n  else:duplicate_hashes[digest]=name\n inventory.append(dict(attribute=name,category=cat,unit=unit,source=source,training_subbasins=len(T),valid_training_subbasins=n+sum(nonnumeric[name].values()),valid_numeric_training_subbasins=n,missing_fraction_numeric_rows=float(1-n/len(T)),missing_fraction_numeric_basin_area_weighted=numeric_missing,missing_fraction_rows=float(np.clip(1-(n+sum(nonnumeric[name].values()))/len(T),0.,1.)),missing_fraction_basin_area_weighted=missing,training_min=mn,training_max=mx,training_weighted_mean=mean,training_weighted_std=sd,constant_training=exact,numerically_near_constant=near,duplicate_of=duplicate_of,domain_role=role,quality_action=action,domain_reason=reason,nonnumeric_training_rows=sum(nonnumeric[name].values()),training_category_counts=json.dumps(dict(nonnumeric[name])),review_flags=\';\'.join(flags)))\nsavecsv(\'attribute_inventory_and_screening.csv\',inventory)\nsavecsv(\'exact_duplicate_candidates.csv\',duplicate_groups,[\'attribute\',\'duplicate_of\',\'unit\'])\nnames={s[1] for s in schema};only=[dict(category=s[0],attribute=s[1],unit=s[2],source=s[3]) for s in lmeta if s[1] not in names];savecsv(\'whole_basin_only_attributes.csv\',only)\n# Diagnostic rank correlations: 8 area-proportional samples per donor; no holdout data used.\nrng=np.random.default_rng(42);selected=[]\nfor b in basins:\n if b[\'catchment_split\']!=\'training_donor\':continue\n lo,hi=b[\'start_row\'],b[\'end_row_exclusive\'];a=X[lo:hi,jarea].astype(\'float64\');cdf=np.cumsum(a/a.sum());u=(np.arange(8)+rng.random(8))/8;selected.extend((lo+np.searchsorted(cdf,u)).tolist())\nS=X[np.array(selected)].astype(\'float64\');usable=[]\nfor j in range(S.shape[1]):\n v=S[:,j];g=np.isfinite(v)\n if g.sum()<2:continue\n v[~g]=np.median(v[g]);unique,inv,counts=np.unique(v,return_inverse=True,return_counts=True)\n if len(unique)<2:continue\n ends=np.cumsum(counts);ranks=(ends-counts+(ends-1))/2.;v[:]=ranks[inv];v-=v.mean();norm=np.sqrt(np.sum(v*v));v/=norm;usable.append(j)\nZ=S[:,usable];C=Z.T@Z;pairs=[]\nfor a,b in zip(*np.where(np.triu(np.abs(C)>=.98,1))):\n i,j=usable[a],usable[b];pairs.append(dict(attribute_1=schema[i][1],attribute_2=schema[j][1],spearman_r=float(C[a,b]),category_1=schema[i][0],category_2=schema[j][0],diagnostic=\'training_only_balanced_area_sample_not_automatic_removal\'))\nsavecsv(\'high_correlation_review_pairs.csv\',pairs,[\'attribute_1\',\'attribute_2\',\'spearman_r\',\'category_1\',\'category_2\',\'diagnostic\'])\nsummary=dict(phase=\'exploration_complete\',basins=1426,training_basins=1326,unseen_basins=100,subbasin_records=len(X),training_subbasin_records=int(trainmask.sum()),unseen_subbasin_records=int((~trainmask).sum()),subbasin_attributes=len(schema),whole_basin_attributes=len(lmeta),whole_basin_only_attributes=len(only),subbasin_count_min=min(b[\'subbasins\'] for b in basins),subbasin_count_max=max(b[\'subbasins\'] for b in basins),single_subbasin_basins=sum(b[\'single_subbasin\'] for b in basins),area_sum_relative_error_max=max(abs(b[\'area_relative_difference\']) for b in basins),categories=dict(collections.Counter(s[0] for s in schema)),sources=dict(collections.Counter(s[3] for s in schema)),quality_actions=dict(collections.Counter(r[\'quality_action\'] for r in inventory)),domain_roles=dict(collections.Counter(r[\'domain_role\'] for r in inventory)),exact_duplicate_candidates=len(duplicate_groups),categorical_attributes=sum(bool(c) for c in nonnumeric.values()),categorical_value_records=len(bad),masked_numeric_records=len(masked),masked_numeric_attributes=len({r["attribute"] for r in masked}),correlation_review_pairs=len(pairs),correlation_absolute_threshold=.98,correlation_training_sample_size=len(selected),pca_fit=False,seconds=round(time.time()-start))\nsavejson(\'exploration_summary.json\',summary);savejson(\'fit_scope.json\',dict(training_basin_ids=fitids,unseen_basin_ids=holdids,fitted_pca=False,quality_statistics_scope=\'training_basins_only\',fit_weight=\'within_basin_area_fraction / 1326; equal total mass per training basin\',preserve_source_data=True))\nprint(json.dumps(summary,indent=2),flush=True)\n'

In [7]:
RUN_FULL_AUDIT=False
if RUN_FULL_AUDIT:
 exec(compile(AUDIT_SOURCE,'subbasin_audit','exec'))